<a href="https://colab.research.google.com/github/Zanzotta/WhaleCall/blob/main/SNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 3.4 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn

import snntorch as snn
from snntorch import surrogate

In [ ]:
class SpikingWhale(nn.Module):
    def __init__(self, beta=0.9, n_classes=2, num_steps=25):
        super().__init__()
        sg = surrogate.fast_sigmoid(slope=num_steps)

        # conv 1D sull'asse delle bande mel; ogni frame è un time step
        self.conv1 = nn.Conv1d(1, 16, 3, padding=1)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=sg)
        self.conv2 = nn.Conv1d(16, 32, 3, padding=1)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=sg)
        self.pool = nn.MaxPool1d(2)
        self.fc1 = nn.Linear(32 * 16, 128)          # 64 -> 32 -> 16 dopo due pool
        self.lif3 = snn.Leaky(beta=beta, spike_grad=sg)
        self.fc2 = nn.Linear(128, n_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=sg)

    def forward(self, x):
        # x: [B, 1, 64, 128] -> [T=128, B, 1, 64]
        x = x.squeeze(1).permute(2, 0, 1).unsqueeze(2)

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem_o = self.lif_out.init_leaky()

        out = []
        for xt in x:                                  # xt: [B, 1, 64]
            h = self.pool(self.conv1(xt))
            h, mem1 = self.lif1(h, mem1)
            h = self.pool(self.conv2(h))
            h, mem2 = self.lif2(h, mem2)
            h = self.fc1(h.flatten(1))
            h, mem3 = self.lif3(h, mem3)
            h = self.fc2(h)
            spk, mem_o = self.lif_out(h, mem_o)
            out.append(spk)

        return torch.stack(out).mean(0)               # firing rate [B, n_classes]


device = "cuda" if torch.cuda.is_available() else "cpu"
model = SpikingWhale().to(device)
criterion = nn.MultiMarginLoss(margin=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
class MinimalSNN(nn.Module):
    def __init__(self, beta=0.9, hidden=32, n_classes=2, num_steps=25, n_bandeMel = 64):
        super().__init__()
        sg = surrogate.fast_sigmoid(slope=num_steps)
        self.fc1 = nn.Linear(n_bandeMel, hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=sg)
        self.fc2 = nn.Linear(hidden, n_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=sg) #, learn_beta=True)

    def forward(self, x):
        x = x.squeeze(1).permute(2, 0, 1)              # [T=128, B, 64]
        mem1 = self.lif1.init_leaky()
        mem_o = self.lif_out.init_leaky()
        out = []
        for xt in x:                                   # xt: [B, 64]
            h, mem1 = self.lif1(self.fc1(xt), mem1)
            spk, mem_o = self.lif_out(self.fc2(h), mem_o)
            out.append(spk)
        return torch.stack(out).mean(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MinimalSNN().to(device)
criterion = nn.MultiMarginLoss(margin=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_epoch(loader):
    model.train()
    tot, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        rates = model(x)
        loss = criterion(rates, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
        correct += (rates.argmax(1) == y).sum().item()
        tot += len(y)
    return loss_sum / tot, correct / tot

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Definiamo le specifiche del dataset simulato
num_samples = 100
num_channels = 1
mel_bands = 64
time_frames = 128
num_classes = 2

# Generiamo feature casuali (X) e label binarie casuali (y)
X_dummy = torch.randn(num_samples, num_channels, mel_bands, time_frames)
y_dummy = torch.randint(0, num_classes, (num_samples,))

# Creazione del Dataset e del DataLoader
dataset = TensorDataset(X_dummy, y_dummy)
test_loader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Dataset simulato creato con {num_samples} campioni.")
print(f"Shape di una feature di input: {X_dummy[0].shape}")

Dataset simulato creato con 100 campioni.
Shape di una feature di input: torch.Size([1, 64, 128])


In [ ]:
# Testiamo un singolo passo di addestramento (epoch) usando la funzione train_epoch
try:
    loss, accuracy = train_epoch(test_loader)
    print(f"Test completato con successo!")
    print(f"Loss: {loss:.4f} | Accuracy: {accuracy * 100:.2f}%")
except Exception as e:
    print(f"Errore durante l'esecuzione del test: {e}")

Test completato con successo!
Loss: 0.1503 | Accuracy: 49.00%
